# Détection de Deepfakes avec XceptionNet

Objectif :
- Classifier des images en deux catégories : **Original** ou **Deepfake**
- Utiliser un modèle pré-entraîné **XceptionNet**
- Évaluer les performances et discuter des résultats


In [3]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device utilisé :", device)
print("GPU dispo :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nom du GPU :", torch.cuda.get_device_name(0))


Device utilisé : cuda
GPU dispo : True
Nom du GPU : NVIDIA GeForce RTX 3050 Laptop GPU


In [4]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())


2.5.1+cu121
True


In [5]:
import os
# Replace the ********** with your actual token
os.environ["KAGGLE_API_TOKEN"] = "KGAT_c189f2b72163160a6599409fb7ee79e2"
# Optional: test Kaggle CLI
#!kaggle competitions list

In [6]:
os.listdir("./faceforensics_c23")


['CSVs', 'FF++C32-All-frames.csv', 'FF++C32-Frames']

## 1) Jeu de données FaceForensics++ C23

Nous utilisons :
- Le dataset **FaceForensics++ C23** (Kaggle)
- Deux classes :
    - 0 = Original
    - 1 = Deepfake
- Les données sont décrites dans deux CSV :
    - `Original.csv`
    - `Deepfakes.csv`

Nous gardons la même préparation des données que dans notre projet précédent.


In [9]:
#imports

import pandas as pd
from sklearn.model_selection import train_test_split

from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

#define the directory of the dataset to not rewrite it entirely 
BASE_DIR = "./faceforensics_c23"

#Build the dataframes : one line contain one image, filename, label, features...
df_real = pd.read_csv(os.path.join(BASE_DIR, "CSVs", "Original.csv"))
df_fake = pd.read_csv(os.path.join(BASE_DIR, "CSVs", "Deepfakes.csv"))

#print the first line on the 2 dataframes
print(df_real.head())
print(df_fake.head())


     filename     label                    features  \
0  543_f2.jpg  Original  Real face, authentic frame   
1  913_f1.jpg  Original  Real face, authentic frame   
2  189_f1.jpg  Original  Real face, authentic frame   
3  214_f3.jpg  Original  Real face, authentic frame   
4  743_f1.jpg  Original  Real face, authentic frame   

                                            filepath  label_id  
0  /kaggle/input/faceforensics-extracted-dataset-...         0  
1  /kaggle/input/faceforensics-extracted-dataset-...         0  
2  /kaggle/input/faceforensics-extracted-dataset-...         0  
3  /kaggle/input/faceforensics-extracted-dataset-...         0  
4  /kaggle/input/faceforensics-extracted-dataset-...         0  
         filename      label                           features  \
0  112_892_f0.jpg  Deepfakes  Manipulated face, synthetic frame   
1  546_621_f1.jpg  Deepfakes  Manipulated face, synthetic frame   
2  509_525_f4.jpg  Deepfakes  Manipulated face, synthetic frame   
3  151_225_

In [10]:
#we merge the two dataframes into one by concatenate them

df = pd.concat([df_real, df_fake], ignore_index=True)

#we print the number of line for the label column 
print(df["label"].value_counts())


label
Original     5000
Deepfakes    5000
Name: count, dtype: int64


In [11]:
label_map = {"Original": 0, "Deepfakes": 1} #transform the two classes into 0 and 1
df["target"] = df["label"].map(label_map)   #add the transformation into one column "target"

#now we have one label link to a value 0 or 1


In [12]:
#we need to build the path to the images by this function
def build_path(row):        #pour chaque ligne du dataframe
    return os.path.join(    #assemble the two (deepfakes and original) in a path
        BASE_DIR,
        "FF++C32-Frames",
        row["label"],       # dossier Deepfakes ou Original
        row["filename"]     # nom du fichier
    )

df["path"] = df.apply(build_path, axis=1)
print(df.head())


     filename     label                    features  \
0  543_f2.jpg  Original  Real face, authentic frame   
1  913_f1.jpg  Original  Real face, authentic frame   
2  189_f1.jpg  Original  Real face, authentic frame   
3  214_f3.jpg  Original  Real face, authentic frame   
4  743_f1.jpg  Original  Real face, authentic frame   

                                            filepath  label_id  target  \
0  /kaggle/input/faceforensics-extracted-dataset-...         0       0   
1  /kaggle/input/faceforensics-extracted-dataset-...         0       0   
2  /kaggle/input/faceforensics-extracted-dataset-...         0       0   
3  /kaggle/input/faceforensics-extracted-dataset-...         0       0   
4  /kaggle/input/faceforensics-extracted-dataset-...         0       0   

                                                path  
0  ./faceforensics_c23\FF++C32-Frames\Original\54...  
1  ./faceforensics_c23\FF++C32-Frames\Original\91...  
2  ./faceforensics_c23\FF++C32-Frames\Original\18...  
3  .

## 2) Séparation du dataset

Les données sont séparées en :
- `train` (60%)
- `validation` (20%)
- `test` (20%)

Nous utilisons `stratify` pour conserver la proportion Original/Deepfake.


In [13]:
#spliting the datas
trainval_df, test_df = train_test_split(
    df,
    test_size=0.2,  #we use 80% of the data to train and 20% to test
    stratify=df["target"],      #to keep the same ratio of data to train and test
)
train_df, val_df = train_test_split(
    trainval_df,
    test_size=0.2,                 # 20% de trainval → ~16% du total
    stratify=trainval_df["target"],
)

print(len(train_df), len(val_df), len(test_df))



6400 1600 2000


## 3) Dataset PyTorch

Nous créons un `Dataset` pour :
- lire l’image
- appliquer les transformations
- retourner (image, label)

Aucune modification par rapport à notre premier projet.


In [14]:
from torch.utils.data import Dataset    #to create our own dataset
from PIL import Image   #to open images
import torchvision.transforms as T  #transform function
import torch    #to create the tensors

#we put all the images at the same size 128x128, then transform it into tensors, and normalize each channel into -1,0 or 1
transform = T.Compose([
    T.Resize((128, 128)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225])
])

#we define our dataset
class DeepfakeDataset(Dataset):
    def __init__(self, df, transform=None):     #df = the dataframe original or fake
        self.df = df.reset_index(drop=True)        #reset the index to zero because at each epoch it updates
        self.transform = transform      #la fonction de normalisation qu'on avait défini plus haut et qu'on va appliquer aux images a chaque époque

    #renvoie le nombre total d'images dans le dataset pour savoir quand s'arreter
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]     #récupère la ligne en fonction de l'itération
        img = Image.open(row["path"]).convert("RGB")    #ouvre l'image via son chemin et vérifie la présence de 3 canaux RGB
        
        if self.transform:      #on applique la normalisation sur l'image
            img = self.transform(img)
        
        label = torch.tensor(row["target"], dtype=torch.long)   #on crée le tensor, de type entier long, a partir du label 0 et 1
        return img, label   #retourne l'image et son label


## 4) Transformations et DataLoader

Transformations :
- `Resize(299x299)` : taille d’entrée pour Xception
- `RandomHorizontalFlip()` : Data augmentation
- `Normalize()` : normalisation des valeurs

`num_workers=4` :
- charge les images **en parallèle**
- accélère l’entraînement


In [15]:
from torch.utils.data import DataLoader     #importer la classe 


#on crée les 3 datasets pour train et test et on applique le transform aux deux
train_dataset = DeepfakeDataset(train_df, transform)    #input : dataframe + normalisation
val_dataset   = DeepfakeDataset(val_df,  transform)
test_dataset  = DeepfakeDataset(test_df, transform)

#On crée les loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)   #input : 1 dataset, la taille 32 images par batch et on active le mélange
                                                                        #pour un meilleur apprentissage
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)   #pareil mais on active pas le mélange pour l'évaluation



## 5) Modèle pré-entraîné : XceptionNet

Différences avec notre CNN de base :
- Nous **n’écrivons plus les couches à la main**
- Nous utilisons un modèle **pré-entraîné sur ImageNet**
- Ce modèle est **SOTA (State Of The Art)**

Modifications faites :
- On **remplace juste la dernière couche**
- Au lieu de 1000 sorties (ImageNet), nous avons **2 classes**


## 6) Optimisation utilisée

Fonction de perte :
- `CrossEntropyLoss` (pareil que dans notre CNN)

Optimiseur :
- `AdamW` : meilleure version de Adam pour les grands modèles

Régularisation :
- `weight_decay` (régularisation L2)
- évite que les poids deviennent trop grands
- réduit l’overfitting

Scheduler :
- `ReduceLROnPlateau`
- baisse automatiquement le learning rate si le modèle stagne

Early stopping :
- arrête l’entraînement si le modèle n’améliore plus la validation
- évite de surentraîner
- fait gagner du temps


## 7) Entraînement

Nous entraînons le modèle sur GPU (RTX 3050).
Durée d'entraînement ≈ 13 minutes.

Nous récupérons :
- la perte (loss)
- l'accuracy (entrainement + validation)


In [19]:
for epoch in range(num_epochs):
    # ----- TRAIN -----
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        _, preds = torch.max(outputs, 1)
        running_correct += (preds == labels).sum().item()
        running_total += labels.size(0)

    train_loss = running_loss / running_total
    train_acc = running_correct / running_total * 100

    # ----- VALIDATION -----
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * labels.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total * 100

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train loss: {train_loss:.4f} | Train acc: {train_acc:.2f}% "
        f"| Val loss: {val_loss:.4f} | Val acc: {val_acc:.2f}%"
    )

    # ----- SCHEDULER + EARLY STOPPING -----
    scheduler.step(val_loss)

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), best_model_path)
        print(f"--> Nouveau meilleur modèle sauvegardé (val_loss = {best_val_loss:.4f})")
    else:
        epochs_no_improve += 1
        print(f"Aucune amélioration depuis {epochs_no_improve} époque(s).")
        if epochs_no_improve >= patience:
            print(f"EARLY STOPPING à l'époque {epoch+1}")
            break


Epoch [1/10] Train loss: 1.7177 | Train acc: 44.75% | Val loss: 0.7130 | Val acc: 53.06%
--> Nouveau meilleur modèle sauvegardé (val_loss = 0.7130)
Epoch [2/10] Train loss: 0.6371 | Train acc: 62.30% | Val loss: 0.6288 | Val acc: 61.94%
--> Nouveau meilleur modèle sauvegardé (val_loss = 0.6288)
Epoch [3/10] Train loss: 0.4911 | Train acc: 74.70% | Val loss: 0.6378 | Val acc: 66.81%
Aucune amélioration depuis 1 époque(s).
Epoch [4/10] Train loss: 0.3365 | Train acc: 83.73% | Val loss: 0.5022 | Val acc: 75.62%
--> Nouveau meilleur modèle sauvegardé (val_loss = 0.5022)
Epoch [5/10] Train loss: 0.2484 | Train acc: 88.30% | Val loss: 0.5023 | Val acc: 76.56%
Aucune amélioration depuis 1 époque(s).
Epoch [6/10] Train loss: 0.1731 | Train acc: 92.33% | Val loss: 0.5417 | Val acc: 79.19%
Aucune amélioration depuis 2 époque(s).
Epoch [7/10] Train loss: 0.1170 | Train acc: 94.86% | Val loss: 0.5317 | Val acc: 80.50%
Aucune amélioration depuis 3 époque(s).
EARLY STOPPING à l'époque 7


In [20]:
# Charger le meilleur modèle sauvegardé
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.to(device)
model.eval()        #re switch to evaluation mode

test_loss = 0.0
test_correct = 0
test_total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        test_loss += loss.item() * labels.size(0)
        _, preds = torch.max(outputs, 1)
        test_correct += (preds == labels).sum().item()
        test_total += labels.size(0)

avg_test_loss = test_loss / test_total
avg_test_acc = test_correct / test_total * 100

print(f"Test loss: {avg_test_loss:.4f} - Test accuracy: {avg_test_acc:.2f}%")


C:\Users\klora\AppData\Local\Temp\ipykernel_39920\2646099867.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path, map_locati

Test loss: 0.5016 - Test accuracy: 75.60%


In [21]:
print(train_df.iloc[0])


filename                                           373_f2.jpg
label                                                Original
features                           Real face, authentic frame
filepath    /kaggle/input/faceforensics-extracted-dataset-...
label_id                                                    0
target                                                      0
path        ./faceforensics_c23\FF++C32-Frames\Original\37...
Name: 4643, dtype: object


## 8) Évaluation et métriques

Nous testons la **meilleure version** du modèle.

Métriques utilisées :
- Test accuracy
- Matrice de confusion
- Précision, Recall, F1-score (classification report)

Pourquoi la matrice de confusion ?
- Elle montre *où* le modèle se trompe, pas juste combien.


In [22]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

all_labels = []
all_preds = []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

all_labels = np.array(all_labels)
all_preds = np.array(all_preds)

cm = confusion_matrix(all_labels, all_preds)
print("Matrice de confusion :")
print(cm)

print("\nRapport de classification :")
print(classification_report(all_labels, all_preds, target_names=["Original", "Deepfake"]))


Matrice de confusion :
[[811 189]
 [299 701]]

Rapport de classification :
              precision    recall  f1-score   support

    Original       0.73      0.81      0.77      1000
    Deepfake       0.79      0.70      0.74      1000

    accuracy                           0.76      2000
   macro avg       0.76      0.76      0.76      2000
weighted avg       0.76      0.76      0.76      2000



## 9) Interprétation des résultats

- Accuracy sur le test ≈ 75.6%
- Matrice de confusion montre que le modèle détecte :
    - 811 originaux correctement
    - 701 deepfakes correctement
- Le modèle est meilleur pour reconnaître les originaux
- Certaines classes de deepfakes restent difficiles


## 10) Lien avec l’article FaceForensics++

Les auteurs montrent que :
- la détection des deepfakes dépend du type de manipulation
- la compression vidéo réduit la performance
- XceptionNet obtient de bonnes performances mais pas parfaites

Nos résultats confirment ce point :
- notre modèle fonctionne bien
- mais certains deepfakes restent difficiles
